# Imports

In [1]:
import sys

sys.path.append("/workspaces/super-duper-dollop/")
import torch
from src.models.CNN import CNN_Simple
from src.data.speaker_dataset import SpeakerDataset
from torch.utils.data import random_split
from src.train_utils import evaluate
import torch.nn as nn
from torch.utils.data import DataLoader

# Model

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN_Simple()
checkpoint = torch.load(
    "/workspaces/super-duper-dollop/src/best_model_cnn_simple.pt",
    map_location=device,
    weights_only=True,
)

In [ ]:
print(model)

CNN_Simple(
  (cnn): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.01)
    (3): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): LeakyReLU(negative_slope=0.01)
    (7): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  )
  (global_pool): AdaptiveAvgPool2d(output_size=(1, 1))
  (fc): Sequential(
    (0): Dropout(p=0.3, inplace=False)
    (1): Linear(in_features=64, out_features=128, bias=True)
    (2): LeakyReLU(negative_slope=0.01)
    (3): Linear(in_features=128, out_features=1, bias=True)
  )
)


In [13]:
dataset = SpeakerDataset(parquet_file="../data/processed/dataset.parquet")
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42),
)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
loss_fn = nn.BCEWithLogitsLoss()

In [16]:
evaluate(model, test_loader, loss_fn, device)

(0.5031357606252035, 0.7931034482758621)